In [ ]:
from pathlib import Path
from datetime import timedelta
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import numpy as np

from mutagen.mp3 import MP3

DATA_DIR = Path("C:\\Users\\edwar\\OneDrive\\Uni\\SoSe2025\\Data Science for Linguists\\Project\\linguistic-data-science\\data\\podcast_data")          # ← root folder that contains one subfolder per podcast
OUT_TABLE = Path("podcast_table.png")
OUT_PLOT  = Path("podcast_plot.png")


def seconds_to_hms(seconds: float) -> str:
    td = timedelta(seconds=int(seconds))
    total_seconds = int(td.total_seconds())
    h = total_seconds // 3600
    m = (total_seconds % 3600) // 60
    s = total_seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def count_words(text: str) -> int:
    return len(text.split())


def audio_duration_seconds(mp3_path: Path) -> float:
    try:
        audio = MP3(str(mp3_path))
        return audio.info.length
    except Exception:
            return 0.0
    return 0.0


def analyze(data_dir: Path) -> list[dict]:
    rows = []
    for podcast_dir in sorted(data_dir.iterdir()):
        if not podcast_dir.is_dir():
            continue

        name       = podcast_dir.name
        n_episodes = 0
        total_words = 0
        total_secs  = 0.0

        txt_files = list(podcast_dir.rglob("*.txt"))
        mp3_files = list(podcast_dir.rglob("*.mp3"))

        for txt in txt_files:
            try:
                text = txt.read_text(encoding="utf-8", errors="ignore")
                total_words += count_words(text)
                n_episodes  += 1
            except Exception as e:
                print(f"  Warning: could not read {txt}: {e}")


        for mp3 in mp3_files:
            total_secs += audio_duration_seconds(mp3)

        if n_episodes == 0:
            continue

        rows.append({
            "podcast":       name,
            "episodes":      n_episodes,
            "words":         total_words,
            "words_per_ep":  total_words / n_episodes,
            "hours_sec":     total_secs,
            "hours_fmt":     seconds_to_hms(total_secs),
        })

    return rows


def make_table_figure(rows: list[dict], out_path: Path):
    total_episodes  = sum(r["episodes"]  for r in rows)
    total_words     = sum(r["words"]     for r in rows)
    total_secs      = sum(r["hours_sec"] for r in rows)

    col_labels = ["Podcasts", "Episodes", "Words", "Words / ep", "Hours\u1d43"]

    table_data = []
    for r in rows:
        table_data.append([
            r["podcast"],
            f"{r['episodes']}",
            f"{r['words']:,}",
            f"{r['words_per_ep']:,.0f}",
            r["hours_fmt"],
        ])

    table_data.append([
        "Total",
        f"{total_episodes}",
        f"{total_words:,}",
        f"{total_words / total_episodes:,.0f}",
        seconds_to_hms(total_secs),
    ])

    n_rows = len(table_data)
    n_cols = len(col_labels)

    fig_h = 0.38 * (n_rows + 2)
    fig, ax = plt.subplots(figsize=(10, max(fig_h, 3)))
    ax.axis("off")


    col_widths = [0.30, 0.12, 0.16, 0.16, 0.16]   
    x_positions = [0]
    for w in col_widths[:-1]:
        x_positions.append(x_positions[-1] + w)

    row_h   = 1.0 / (n_rows + 2)  
    header_y = 1.0 - row_h          


    ax.axhline(1.0, color="black", linewidth=1.4, xmin=0, xmax=1)
    ax.axhline(header_y - 0.005, color="black", linewidth=0.8, xmin=0, xmax=1)
    totals_y = row_h + 0.005
    ax.axhline(totals_y, color="black", linewidth=0.8, xmin=0, xmax=1)
    ax.axhline(0.0, color="black", linewidth=1.4, xmin=0, xmax=1)

    align = ["left", "center", "right", "right", "right"]
    for c, (label, x, al) in enumerate(zip(col_labels, x_positions, align)):
        ax.text(x + (col_widths[c] * (0 if al == "left" else 0.5 if al == "center" else 1)),
                header_y + row_h * 0.4,
                label,
                ha=al, va="center",
                fontsize=9, fontweight="bold",
                fontfamily="DejaVu Sans")

    for r_idx, row_vals in enumerate(table_data):
        is_total = r_idx == len(table_data) - 1
        y = header_y - (r_idx + 1) * row_h + row_h * 0.35
        for c, (val, x, al) in enumerate(zip(row_vals, x_positions, align)):
            ax.text(x + (col_widths[c] * (0 if al == "left" else 0.5 if al == "center" else 1)),
                    y,
                    val,
                    ha=al, va="center",
                    fontsize=8.5,
                    fontstyle="italic" if (not is_total and c == 0) else "normal",
                    fontfamily="DejaVu Sans")

    ax.text(0, -row_h * 0.6,
            "\u1d43 Format of time units: hours:minutes:seconds",
            ha="left", va="center", fontsize=7.5, color="#444",
            fontfamily="DejaVu Sans")


    ax.text(0, 1.0 + row_h * 0.6,
            "Table 1.  General information about podcast corpus.",
            ha="left", va="center", fontsize=9.5, fontweight="bold",
            fontfamily="DejaVu Sans")

    ax.set_xlim(0, 1)
    ax.set_ylim(-row_h, 1.0 + row_h)

    fig.tight_layout(pad=0.5)
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    print(f"Table saved → {out_path}")


def make_plot_figure(rows: list[dict], out_path: Path):
    """Render a 2-panel bar chart (words + hours) and save as PNG."""
    names  = [r["podcast"]    for r in rows]
    words  = [r["words"]      for r in rows]
    hours  = [r["hours_sec"] / 3600 for r in rows]

    x = np.arange(len(names))
    bar_w = 0.55

    c_words = "#2E6DA4"
    c_hours = "#E07B39"

    fig = plt.figure(figsize=(11, 6))
    gs  = GridSpec(1, 2, figure=fig, wspace=0.38)


    ax1 = fig.add_subplot(gs[0])
    bars1 = ax1.bar(x, words, width=bar_w, color=c_words, edgecolor="white", linewidth=0.6)
    ax1.set_xticks(x)
    ax1.set_xticklabels(names, rotation=35, ha="right", fontsize=8.5)
    ax1.set_ylabel("Number of Words", fontsize=9)
    ax1.set_title("(a) Word Count per Podcast", fontsize=10, pad=8)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v/1000:.0f}k"))
    ax1.spines[["top", "right"]].set_visible(False)
    ax1.tick_params(axis="y", labelsize=8)
    ax1.bar_label(bars1, labels=[f"{w/1000:.0f}k" for w in words],
                  padding=3, fontsize=7, color="#333")


    ax2 = fig.add_subplot(gs[1])
    bars2 = ax2.bar(x, hours, width=bar_w, color=c_hours, edgecolor="white", linewidth=0.6)
    ax2.set_xticks(x)
    ax2.set_xticklabels(names, rotation=35, ha="right", fontsize=8.5)
    ax2.set_ylabel("Audio Duration (hours)", fontsize=9)
    ax2.set_title("(b) Total Audio Duration per Podcast", fontsize=10, pad=8)
    ax2.spines[["top", "right"]].set_visible(False)
    ax2.tick_params(axis="y", labelsize=8)
    ax2.bar_label(bars2, labels=[f"{h:.1f}h" for h in hours],
                  padding=3, fontsize=7, color="#333")

    fig.suptitle("Podcast Corpus Overview", fontsize=12, fontweight="bold", y=1.01)
    fig.tight_layout()
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    print(f"Plot saved  → {out_path}")


if __name__ == "__main__":

    rows = analyze(DATA_DIR)


    print(f"{'Podcast':<22} {'Episodes':>9} {'Words':>10} {'Wds/ep':>8} {'Duration':>10}")
    print("-" * 62)
    for r in rows:
        print(f"{r['podcast']:<22} {r['episodes']:>9} {r['words']:>10,} "
              f"{r['words_per_ep']:>8,.0f} {r['hours_fmt']:>10}")
    total_ep = sum(r["episodes"] for r in rows)
    total_w  = sum(r["words"]    for r in rows)
    total_s  = sum(r["hours_sec"] for r in rows)
    print("-" * 62)
    print(f"{'Total':<22} {total_ep:>9} {total_w:>10,} "
          f"{total_w/total_ep:>8,.0f} {seconds_to_hms(total_s):>10}\n")

    make_table_figure(rows, OUT_TABLE)
    make_plot_figure(rows,  OUT_PLOT)


Scanning podcast corpus …

Podcast                 Episodes      Words   Wds/ep   Duration
--------------------------------------------------------------
alles_geschichte              10     35,897    3,590   04:40:14
auf_der_spur_-_ard_ermittlerkrimis        10     55,169    5,517   06:43:31
baywatch_berlin               10    139,666   13,967   12:08:49
edeltalk                      10    117,319   11,732   09:06:44
forschergeist                 10    136,670   13,667   15:45:47
klinisch_relevant_podcast        10     39,398    3,940   04:27:26
lage_der_nation                9    161,179   17,909   15:25:01
swr_kultur_lesenswert_-_literatur        10     30,190    3,019   04:10:23
zmethodisch_inkorrekt         10    189,387   18,939   20:19:18
ö1_radiokolleg                10     23,611    2,361   02:46:49
--------------------------------------------------------------
Total                         99    928,486    9,379   95:34:08

Table saved → podcast_table.png


C:\Users\edwar\AppData\Local\Temp\ipykernel_104304\1316920701.py:208: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


Plot saved  → podcast_plot.png
